# `gain_skeletons` demonstrator

`gain_skeletons` builds mock `xarray` datasets that scaffold radio
interferometric calibration ("gain") solutions, writes them to zarr, and
reads them back. **It is a demonstrator: every value in every dataset below
is randomly generated, and nothing in this package computes or applies
calibration.**

It ships a catalogue of eleven calibration types, split into the
direction-independent ones (phenomenological gain, antenna gain,
tropospheric gain, opacity, bandpass, leakage, delay, antenna positions,
fringe fit) and the direction-dependent ones (direction-dependent
phenomenological gain, ionosphere). Keys are spelled out rather than
abbreviated to the single letters convention assigns some of them. The
catalogue is illustrative rather than exhaustive: it exists to cover the
range of coordinate shapes these datasets take, and the last section shows
that a type it does not carry needs no change to the package. This notebook
walks through the catalogue using only the package's public API, so every
dataset shown here comes from `gain_skeletons` itself; the notebook defines
no schema of its own.

## Imports and the catalogue

`gain_skeletons` is imported under the conventional short alias `gs`.
`list_cal_types` returns the registry keys, direction-independent types
first.

In [1]:
import numpy as np
import xarray as xr

import gain_skeletons as gs

gs.list_cal_types()

('phenomenological_gain',
 'antenna_gain',
 'tropospheric_gain',
 'opacity',
 'bandpass',
 'leakage',
 'delay',
 'antenna_positions',
 'fringe_fit',
 'dd_phenomenological_gain',
 'ionosphere')

## Coordinate factories on their own

Each axis the package uses has a standalone factory function that returns a
one-dimensional `xarray.DataArray`. They are usable independently of any
dataset, and their default ranges (a MeerKAT L-band frequency span, a
fixed time origin) are overridable via keyword arguments.

In [2]:
print(gs.time_coord(3, start=0.0, interval=8.0).values)
print(gs.frequency_coord(5).values)
print(gs.frequency_coord(5, start=1.0e9, end=2.0e9).values)
print(gs.antenna_name_coord(4).values)
gs.frequency_coord(4)

[ 0.  8. 16.]
[8.560e+08 1.070e+09 1.284e+09 1.498e+09 1.712e+09]
[1.00e+09 1.25e+09 1.50e+09 1.75e+09 2.00e+09]
['m000' 'm001' 'm002' 'm003']


<xarray.DataArray 'frequency' (frequency: 4)> Size: 32B
array([8.56000000e+08, 1.14133333e+09, 1.42666667e+09, 1.71200000e+09])
Dimensions without coordinates: frequency
Attributes:
    type:                 spectral_coord
    units:                Hz
    observer:             topo
    reference_frequency:  856000000.0
    channel_width:        285333333.3333333

## The simplest case, `antenna_gain`

The standard electronic gain is a complex, on-diagonal-only gain with one
solution for the whole band. Read the dataset repr below against that
description: four axes, a single complex `GAIN` array, units of `rel`, and a
frequency axis of length one — present, but unresolved, which is not the
same thing as absent.

In [3]:
gs.make_gain_xds("antenna_gain")

<xarray.Dataset> Size: 720B
Dimensions:         (time: 4, antenna_name: 8, frequency: 1, receptor_label: 2)
Coordinates:
  * time            (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name    (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency       (frequency) float64 8B 8.56e+08
  * receptor_label  (receptor_label) <U1 8B 'X' 'Y'
Data variables:
    GAIN            (time, antenna_name, frequency, receptor_label) complex64 512B ...
    FLAG            (time, antenna_name, frequency) bool 32B False ... False
Attributes:
    cal_type:             antenna_gain
    direction_dependent:  False
    jones_structure:      diagonal
    description:          Standard electronic gain, on-diagonal only, one sol...

## `bandpass` against `antenna_gain`: channel-resolved against single-channel

`bandpass` has the same axis list as `antenna_gain`, but is resolved per
channel rather than carrying one solution for the whole band. Both are
on-diagonal-only complex gains, so their `GAIN` arrays share the same
dimensions; only the frequency axis's extent differs.

In [4]:
bandpass = gs.make_gain_xds("bandpass", n_frequency=64)
antenna_gain = gs.make_gain_xds("antenna_gain")
print("bandpass dims    :", dict(bandpass.sizes))
print("antenna_gain dims:", dict(antenna_gain.sizes))
print("same axes:", bandpass.GAIN.dims == antenna_gain.GAIN.dims)
bandpass

bandpass dims    : {'time': 4, 'antenna_name': 8, 'frequency': 64, 'receptor_label': 2}
antenna_gain dims: {'time': 4, 'antenna_name': 8, 'frequency': 1, 'receptor_label': 2}
same axes: True


<xarray.Dataset> Size: 35kB
Dimensions:         (time: 4, antenna_name: 8, frequency: 64, receptor_label: 2)
Coordinates:
  * time            (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name    (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency       (frequency) float64 512B 8.56e+08 8.696e+08 ... 1.712e+09
  * receptor_label  (receptor_label) <U1 8B 'X' 'Y'
Data variables:
    GAIN            (time, antenna_name, frequency, receptor_label) complex64 33kB ...
    FLAG            (time, antenna_name, frequency) bool 2kB False ... False
Attributes:
    cal_type:             bandpass
    direction_dependent:  False
    jones_structure:      diagonal
    description:          Standard bandpass, on-diagonal only, resolved in fr...

## `phenomenological_gain`: the generic case

A `phenomenological_gain` is a generic representation of any 2x2 Jones matrix. These are useful
as an interchange format as they don't need to be evaluated. These currently use
`gain_X` and `gain_Y` as their parameter labels, but this is still up for debate.

In [5]:
complex_gain = gs.make_gain_xds("phenomenological_gain")
complex_gain

<xarray.Dataset> Size: 68kB
Dimensions:          (time: 4, antenna_name: 8, frequency: 64,
                      receptor_label: 2, parameter_label: 2)
Coordinates:
  * time             (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name     (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency        (frequency) float64 512B 8.56e+08 8.696e+08 ... 1.712e+09
  * receptor_label   (receptor_label) <U1 8B 'X' 'Y'
  * parameter_label  (parameter_label) <U6 48B 'gain_X' 'gain_Y'
Data variables:
    GAIN             (time, antenna_name, frequency, receptor_label, parameter_label) complex64 66kB ...
    FLAG             (time, antenna_name, frequency) bool 2kB False ... False
Attributes:
    cal_type:             phenomenological_gain
    direction_dependent:  False
    jones_structure:      full
    description:          General Jones term, describing the response without...

## `antenna_positions`: axes genuinely absent, and meaningful parameter labels

`antenna_positions` is neither frequency- nor polarisation-dependent, so it
carries no `frequency` and no `receptor_label` axis whatsoever. An absent
axis is materially different from an axis that exists with length one, such
as `antenna_gain`'s frequency axis above: the first says the quantity has no
such dependence, the second says it has one solution across that dependence.
Its three same-unit components — an antenna position offset in each of X, Y
and Z — instead occupy a `parameter_label` axis.

In [6]:
antenna_positions = gs.make_gain_xds("antenna_positions")
print("axes present:", antenna_positions.ANTENNA_POSITION_OFFSET.dims)
print("frequency absent:", "frequency" not in antenna_positions.dims)
print("parameter labels:", list(antenna_positions.parameter_label.values))
antenna_positions

axes present: ('time', 'antenna_name', 'parameter_label')
frequency absent: True
parameter labels: [np.str_('dX'), np.str_('dY'), np.str_('dZ')]


<xarray.Dataset> Size: 984B
Dimensions:                  (time: 4, antenna_name: 8, parameter_label: 3)
Coordinates:
  * time                     (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name             (antenna_name) <U4 128B 'm000' 'm001' ... 'm007'
  * parameter_label          (parameter_label) <U2 24B 'dX' 'dY' 'dZ'
Data variables:
    ANTENNA_POSITION_OFFSET  (time, antenna_name, parameter_label) float64 768B ...
    FLAG                     (time, antenna_name) bool 32B False False ... False
Attributes:
    cal_type:             antenna_positions
    direction_dependent:  False
    description:          Antenna position correction. Three same-unit compon...

## `ionosphere`: the direction axis

`ionosphere` is one of the two direction-dependent types. `direction` is an
integer index into a direction list held elsewhere (a facet within one MSv4
field of view), not a sky position itself. The axis appears only for
calibration types with genuine direction dependence; none of the
direction-independent types carry it.

In [7]:
gs.make_gain_xds("ionosphere", n_direction=4)

<xarray.Dataset> Size: 1kB
Dimensions:       (direction: 4, time: 4, antenna_name: 8)
Coordinates:
  * direction     (direction) int64 32B 0 1 2 3
  * time          (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name  (antenna_name) <U4 128B 'm000' 'm001' 'm002' ... 'm006' 'm007'
Data variables:
    TEC           (direction, time, antenna_name) float64 1kB 0.6287 ... 1.292
    FLAG          (direction, time, antenna_name) bool 128B False ... False
Attributes:
    cal_type:             ionosphere
    direction_dependent:  True
    description:          Ionospheric total electron content. Direction-depen...

## `delay`: a parameterised type

`delay` stores the parameters of a phase ramp across frequency — an offset
in degrees and a slope in seconds — rather than the ramp itself sampled
channel by channel. Two things follow from that. Its `frequency` axis is
present but single-channel, since one ramp is solved for the whole band; and
because its two quantities carry different units, the two layouts have
something to disagree about for the first time in this notebook.

Consolidated, the two quantities share one `PARAMETER` array and units move
to a `parameter_units` coordinate. Split, each is its own array carrying its
own scalar `units`. Both quantities are polarised, so nothing has to be
broadcast either way — that cost appears only with fringe fit, below.

In [8]:
delay = gs.make_gain_xds("delay")
print("axes present    :", delay.PARAMETER.dims)
print("frequency extent:", delay.sizes["frequency"])
print("parameter labels:", list(delay.parameter_label.values))
print("parameter units :", list(delay.parameter_units.values))
delay

axes present    : ('time', 'antenna_name', 'frequency', 'receptor_label', 'parameter_label')
frequency extent: 1
parameter labels: [np.str_('PHASE'), np.str_('DELAY')]
parameter units : [np.str_('deg'), np.str_('s')]


<xarray.Dataset> Size: 1kB
Dimensions:          (time: 4, antenna_name: 8, frequency: 1,
                      receptor_label: 2, parameter_label: 2)
Coordinates:
  * time             (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name     (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency        (frequency) float64 8B 8.56e+08
  * receptor_label   (receptor_label) <U1 8B 'X' 'Y'
  * parameter_label  (parameter_label) <U5 40B 'PHASE' 'DELAY'
    parameter_units  (parameter_label) <U3 24B 'deg' 's'
Data variables:
    PARAMETER        (time, antenna_name, frequency, receptor_label, parameter_label) float64 1kB ...
    FLAG             (time, antenna_name, frequency) bool 32B False ... False
Attributes:
    cal_type:             delay
    direction_dependent:  False
    description:          Delay. A phase offset and a slope in seconds per re...

## Fringe fit both ways

Fringe fit is the larger of the catalogue's two multi-parameter types —
`PHASE`, `DELAY`, `RATE` and `DISP_DELAY`, all from one solve — and the only
one whose quantities disagree about whether they are polarised.
`DISP_DELAY` carries no `receptor_label` axis; the other three do. That
disagreement is what makes the choice of layout cost something here.
`gain_skeletons` offers both, and privileges neither as correct. Both
produce one dataset for the solve, carrying one `FLAG`; what differs is how
that dataset holds the four quantities:

- **Consolidated** (`make_gain_xds`): all four share one `PARAMETER` array,
  indexed by an explicit `parameter_label` axis. This keeps every parameter
  needed to describe one solve adjacent in memory and, once written, in one
  chunked zarr array rather than four that chunk and compress
  independently. The cost is that `DISP_DELAY`, which is unpolarised, must
  be broadcast redundantly across the `receptor_label` axis to sit in the
  same array as the three polarised quantities, and that units move to a
  coordinate.
- **Split** (`make_split_gain_xds`): each quantity is its own array within
  the dataset, named for it, with exactly the axes it needs — `DISP_DELAY`
  keeps no `receptor_label` at all — and a scalar `units` attribute, since
  each array has exactly one unit. The cost is that the four are no longer
  adjacent.

Note what the split layout does not carry: no `parameter_label` axis. In the
consolidated layout that axis is what tells `PHASE` from `DELAY`; split,
each array's name does that job already, so an axis of length one restating
it would say nothing. Where the axis distinguishes components *within* one
quantity — `antenna_positions`' `dX`, `dY` and `dZ` above — it survives in
both layouts.

In [9]:
consolidated = gs.make_gain_xds("fringe_fit")
split = gs.make_split_gain_xds("fringe_fit")

print("consolidated arrays:", list(consolidated.data_vars))
print("split arrays       :", list(split.data_vars))
print("consolidated dims  :", dict(consolidated.sizes))
print("split dims         :", dict(split.sizes))
consolidated

consolidated arrays: ['PARAMETER', 'FLAG']
split arrays       : ['PHASE', 'DELAY', 'RATE', 'DISP_DELAY', 'FLAG']
consolidated dims  : {'time': 4, 'antenna_name': 8, 'frequency': 1, 'receptor_label': 2, 'parameter_label': 4}
split dims         : {'time': 4, 'antenna_name': 8, 'frequency': 1, 'receptor_label': 2}


<xarray.Dataset> Size: 2kB
Dimensions:          (time: 4, antenna_name: 8, frequency: 1,
                      receptor_label: 2, parameter_label: 4)
Coordinates:
  * time             (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name     (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency        (frequency) float64 8B 8.56e+08
  * receptor_label   (receptor_label) <U1 8B 'X' 'Y'
  * parameter_label  (parameter_label) <U10 160B 'PHASE' ... 'DISP_DELAY'
    parameter_units  (parameter_label) <U3 48B 'deg' 's' 's/s' 's'
Data variables:
    PARAMETER        (time, antenna_name, frequency, receptor_label, parameter_label) float64 2kB ...
    FLAG             (time, antenna_name, frequency) bool 32B False ... False
Attributes:
    cal_type:             fringe_fit
    direction_dependent:  False
    description:          Fringe fit. Four quantities with differing units, p...

In [10]:
split

<xarray.Dataset> Size: 2kB
Dimensions:         (time: 4, antenna_name: 8, frequency: 1, receptor_label: 2)
Coordinates:
  * time            (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name    (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * frequency       (frequency) float64 8B 8.56e+08
  * receptor_label  (receptor_label) <U1 8B 'X' 'Y'
Data variables:
    PHASE           (time, antenna_name, frequency, receptor_label) float64 512B ...
    DELAY           (time, antenna_name, frequency, receptor_label) float64 512B ...
    RATE            (time, antenna_name, frequency, receptor_label) float64 512B ...
    DISP_DELAY      (time, antenna_name, frequency) float64 256B 1.087e-09 .....
    FLAG            (time, antenna_name, frequency) bool 32B False ... False
Attributes:
    cal_type:             fringe_fit
    direction_dependent:  False
    description:          Fringe fit. Four quantities with differing units, p...

## Units in the consolidated layout

Because fringe fit's four quantities do not share a unit, the consolidated
`PARAMETER` array cannot carry a scalar `units` attribute the way a
single-unit type's array does — that would falsely claim one unit for a
heterogeneous array. Instead, units move to a `parameter_units` coordinate
aligned with `parameter_label`, which travels along with any selection.
`delay` above is described the same way, for the same reason.

In [11]:
print("scalar units attr:", consolidated.PARAMETER.attrs.get("units", "<absent>"))
print("parameter_units  :", list(consolidated.parameter_units.values))
print("selecting DELAY  :", consolidated.sel(parameter_label="DELAY").parameter_units.item())
print(
    "split units      :",
    {name: array.attrs["units"] for name, array in split.data_vars.items() if name != "FLAG"},
)

scalar units attr: <absent>
parameter_units  : [np.str_('deg'), np.str_('s'), np.str_('s/s'), np.str_('s')]
selecting DELAY  : s
split units      : {'PHASE': 'deg', 'DELAY': 's', 'RATE': 's/s', 'DISP_DELAY': 's'}


## The cost of consolidating

This is the redundancy called out above, made concrete: in the consolidated
layout, `DISP_DELAY`'s values are identical across the `receptor_label`
axis, because the same unpolarised value has been broadcast to both
receptors so it can live in the same array as the polarised quantities.
The split layout never introduces this redundancy — `DISP_DELAY` keeps no
`receptor_label` axis at all. The `FLAG` is byte-for-byte the same in both,
which the next section takes up.

In [12]:
disp = consolidated.PARAMETER.sel(parameter_label="DISP_DELAY")
print(
    "DISP_DELAY repeated across receptors:",
    np.array_equal(disp.sel(receptor_label="X").values, disp.sel(receptor_label="Y").values),
)
print("split DISP_DELAY axes :", split.DISP_DELAY.dims)
print("consolidated flag dims:", consolidated.FLAG.dims)
print("split flag dims       :", split.FLAG.dims)
print("flags identical       :", np.array_equal(consolidated.FLAG.values, split.FLAG.values))

DISP_DELAY repeated across receptors: True
split DISP_DELAY axes : ('time', 'antenna_name', 'frequency')
consolidated flag dims: ('time', 'antenna_name', 'frequency')
split flag dims       : ('time', 'antenna_name', 'frequency')
flags identical       : True


## Flagging

Every dataset carries exactly one boolean `FLAG`, in both layouts, and it is
deliberately coarser than the parameter arrays it describes. `FLAG` never
carries `parameter_label` or `receptor_label`, because those two index the
components of a single solution rather than distinct solutions: the
quantities one solve produced, and the receptors it solved together. A
solution with one untrustworthy component is not one whose remaining
components can be relied on.

`time`, `antenna_name`, `frequency` and `direction` do index genuinely
separate solutions, so `FLAG` keeps them. A direction-dependent type
therefore carries a flag one axis wider than a direction-independent one.

In [13]:
for key in ("bandpass", "fringe_fit", "antenna_positions", "dd_phenomenological_gain"):
    xds = gs.make_gain_xds(key)
    array = gs.get_spec(key).resolved_consolidated_name
    print(f"{key}:")
    print(f"  {array:<24} {xds[array].dims}")
    print(f"  {'FLAG':<24} {xds.FLAG.dims}")

bandpass:
  GAIN                     ('time', 'antenna_name', 'frequency', 'receptor_label')
  FLAG                     ('time', 'antenna_name', 'frequency')
fringe_fit:
  PARAMETER                ('time', 'antenna_name', 'frequency', 'receptor_label', 'parameter_label')
  FLAG                     ('time', 'antenna_name', 'frequency')
antenna_positions:
  ANTENNA_POSITION_OFFSET  ('time', 'antenna_name', 'parameter_label')
  FLAG                     ('time', 'antenna_name')
dd_phenomenological_gain:
  GAIN                     ('direction', 'time', 'antenna_name', 'frequency', 'receptor_label', 'parameter_label')
  FLAG                     ('direction', 'time', 'antenna_name', 'frequency')


## Round-trip to zarr and the on-disk layout

Both layouts write to zarr with `consolidated=False` on both write and
read: zarr format 3 does not specify consolidated metadata, so omitting the
flag draws a `ZarrUserWarning` on write and a `RuntimeWarning` on read
(because it hunts for metadata that was never written). Passing
`consolidated=True` on read does not warn — it raises `ValueError` outright,
since it demands metadata that is not there. Everything below lives
in a `tempfile.TemporaryDirectory`, so nothing from this notebook is left
on disk afterwards. The directory listings make the one-array-against-four
difference visible on disk rather than only in memory: a zarr store holds
one directory per array, so the consolidated store has a single `PARAMETER`
directory beside its `parameter_label` and `parameter_units` coordinates,
where the split store has four parameter directories and neither
coordinate.

In [14]:
import tempfile
from pathlib import Path


def show_arrays(path: Path) -> None:
    """Print the arrays a zarr store holds, one directory each."""
    for entry in sorted(p.name for p in path.iterdir() if p.is_dir()):
        print("  ", entry)


with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)

    consolidated.to_zarr(root / "consolidated.zarr", consolidated=False)
    split.to_zarr(root / "split.zarr", consolidated=False)

    print("consolidated store:")
    show_arrays(root / "consolidated.zarr")
    print("\nsplit store:")
    show_arrays(root / "split.zarr")

    reread = xr.open_dataset(root / "consolidated.zarr", engine="zarr", consolidated=False).load()
    print("\nround-trip identical:", reread.identical(consolidated))

consolidated store:
   FLAG
   PARAMETER
   antenna_name
   frequency
   parameter_label
   parameter_units
   receptor_label
   time

split store:
   DELAY
   DISP_DELAY
   FLAG
   PHASE
   RATE
   antenna_name
   frequency
   receptor_label
   time



round-trip identical: True


## The escape hatch

The eleven registered calibration types are a convenience, not a limitation.
`CalSpec` and `ParamSpec` are public, so a calibration type the registry
does not carry — here, an antenna pointing offset — can be hand-written and
passed to `make_gain_xds` exactly like a registry name. The registry is a
catalogue of examples, not the whole of what the package can express.

In [15]:
pointing = gs.CalSpec(
    name="pointing_offset",
    parameters=(
        gs.ParamSpec(
            name="POINTING_OFFSET",
            units="rad",
            axes=("time", "antenna_name", "parameter_label"),
            dtype="float64",
            labels=("dAZ", "dEL"),
            scale=1.0e-4,
        ),
    ),
    default_sizes={"time": 4, "antenna_name": 8},
    description="Antenna pointing correction; not in the registry.",
)

gs.make_gain_xds(pointing)

<xarray.Dataset> Size: 728B
Dimensions:          (time: 4, antenna_name: 8, parameter_label: 2)
Coordinates:
  * time             (time) float64 32B 1.7e+09 1.7e+09 1.7e+09 1.7e+09
  * antenna_name     (antenna_name) <U4 128B 'm000' 'm001' ... 'm006' 'm007'
  * parameter_label  (parameter_label) <U3 24B 'dAZ' 'dEL'
Data variables:
    POINTING_OFFSET  (time, antenna_name, parameter_label) float64 512B 1.257...
    FLAG             (time, antenna_name) bool 32B False False ... False False
Attributes:
    cal_type:             pointing_offset
    direction_dependent:  False
    description:          Antenna pointing correction; not in the registry.